In [1]:
import sys

import matplotlib
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__, matplotlib.__version__)


3.13.3 2.3.4 2.3.3 1.7.2 3.10.7


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 2650
data = pd.read_csv("data/spambase.csv")
feature_names = [c for c in data.columns if c != "spam"]
X, y = data[feature_names].to_numpy(), data["spam"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(X.shape, round(100 * y.mean(), 1), X_train.shape, X_test.shape)

(4204, 57) 39.9 (3153, 57) (1051, 57)


In [3]:
from sklearn.base import clone
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)
bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=SEED)
rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=SEED, n_jobs=-1)
lr = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=2000))
bag_lr = BaggingClassifier(lr, n_estimators=50, random_state=SEED)
knn = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
ada = AdaBoostClassifier(n_estimators=400, random_state=SEED)
members = [("lr", lr), ("knn", knn), ("tree", tree)]

PART 1

In [6]:
def oob_mask(sample_indices, n_samples):
    """Boolean mask of the rows that do NOT occur in sample_indices."""
    mask = np.ones(n_samples, dtype=bool)
    mask[np.asarray(sample_indices, dtype=int)] = False
    return mask

In [7]:
print(oob_mask([0, 2, 2], 5))        # [False  True False  True  True]
print(oob_mask([], 3))               # [ True  True  True]
print(oob_mask(np.array([3, 0]), 4)) # [False  True  True False]

[False  True False  True  True]
[ True  True  True]
[False  True  True False]


In [8]:
bag.fit(X_train, y_train)

n = len(y_train)
oob_shares = [oob_mask(idx, n).mean() for idx in bag.estimators_samples_]

mean_oob = np.mean(oob_shares)
print("mean OOB share:      ", mean_oob)
print("mean in-bag share:   ", 1 - mean_oob)
print("formula (1-1/n)^n:   ", (1 - 1/n) ** n)
print("formula 1-(1-1/n)^n: ", 1 - (1 - 1/n) ** n)

mean OOB share:       0.3680653346019664
mean in-bag share:    0.6319346653980336
formula (1-1/n)^n:    0.3678210954587638
formula 1-(1-1/n)^n:  0.6321789045412363


In [10]:
votes = np.zeros((n, 2), dtype=int)

for t, idx in zip(bag.estimators_, bag.estimators_samples_):
    mask = oob_mask(idx, n)
    rows = np.where(mask)[0]
    pred = t.predict(X_train[rows])
    votes[rows, pred] += 1

oob_pred = votes.argmax(axis=1)
my_oob_acc = (oob_pred == y_train).mean()

print("my OOB accuracy:", my_oob_acc)
print("bag.oob_score_: ", bag.oob_score_)

my OOB accuracy: 0.9365683476054552
bag.oob_score_:  0.9365683476054552


In [12]:
from sklearn.model_selection import cross_validate

models = {"tree": tree, "bag": bag, "rf": rf, "lr": lr, "bag_lr": bag_lr}

fold_scores = {}
for name, model in models.items():
    fold_scores[name] = cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy")["test_score"]

cv_table = pd.DataFrame(fold_scores).T
cv_table.columns = [f"fold {i+1}" for i in range(5)]
cv_table["mean"] = [np.mean(s) for s in fold_scores.values()]
cv_table["std"] = [np.std(s) for s in fold_scores.values()]
print(cv_table.round(4))

        fold 1  fold 2  fold 3  fold 4  fold 5    mean     std
tree    0.8986  0.8796  0.9033  0.8905  0.9032  0.8950  0.0090
bag     0.9334  0.9192  0.9398  0.9413  0.9206  0.9309  0.0093
rf      0.9445  0.9319  0.9572  0.9587  0.9333  0.9451  0.0114
lr      0.9334  0.9319  0.9445  0.9349  0.9349  0.9359  0.0044
bag_lr  0.9366  0.9303  0.9414  0.9333  0.9365  0.9356  0.0037


In [13]:
def paired(a, b):
    d = fold_scores[a] - fold_scores[b]
    inside = abs(d.mean()) <= d.std()
    print(f"{a} - {b}: folds {np.round(d, 4)}, mean {d.mean():.4f}, std {d.std():.4f}, inside the spread: {inside}")

paired("bag", "tree")
paired("rf", "tree")
paired("bag_lr", "lr")

bag - tree: folds [0.0349 0.0396 0.0365 0.0508 0.0175], mean 0.0358, std 0.0107, inside the spread: False
rf - tree: folds [0.046  0.0523 0.0539 0.0683 0.0302], mean 0.0501, std 0.0124, inside the spread: False
bag_lr - lr: folds [ 0.0032 -0.0016 -0.0032 -0.0016  0.0016], mean -0.0003, std 0.0023, inside the spread: True


In [14]:
rf.fit(X_train, y_train)

p = rf.score(X_test, y_test)
se = np.sqrt(p * (1 - p) / len(y_test))

print("rf oob_score_:     ", rf.oob_score_)
print("rf CV mean:        ", fold_scores["rf"].mean(), "(fold std:", fold_scores["rf"].std(), ")")
print("rf test accuracy p:", p, " SE:", se)
print("|oob - cv mean|:   ", abs(rf.oob_score_ - fold_scores["rf"].mean()))

rf oob_score_:      0.9444973041547733
rf CV mean:         0.9451321912811611 (fold std: 0.01137255199473839 )
rf test accuracy p: 0.9514747859181731  SE: 0.0066279773253566715
|oob - cv mean|:    0.0006348871263878175


## Part 1 — conclusions

**Out-of-bag share.** The mean out-of-bag share per tree is 0.3681, and the mean in-bag share is 0.6319. The formulas give (1 − 1/n)^n = 0.3678 and 1 − (1 − 1/n)^n = 0.6322, so the simulation matches the theory. My hand-made out-of-bag accuracy (0.93657) is identical to `bag.oob_score_` (0.93657).

**(a) Did bagging help?**

- **Decision tree: yes.** The paired fold difference bag − tree has mean +0.0358 and std 0.0107. The mean is larger than the std, so the difference is outside the fold spread, and bagging was better on all five folds (CV accuracy 0.8950 → 0.9309). The forest improves it further: rf − tree has mean +0.0501 and std 0.0124, also outside the spread (0.9451).
- **Logistic regression: no.** The difference bag_lr − lr has mean −0.0003 and std 0.0023. The mean is smaller than the std, so it is inside the fold spread, and these data cannot tell the two apart (0.9356 vs 0.9359). No winner is declared.
- **Mechanism.** A fully grown tree has high variance: a small change in the training sample changes the tree a lot. Bagging trains many trees on bootstrap samples and averages their votes, which reduces variance while the bias stays the same. Logistic regression is a stable, low-variance model, so models fitted on different bootstrap samples are almost identical and averaging them has little to remove. This is consistent with the fold std: 0.0090 for the tree and 0.0044 for logistic regression.

**(b) Is the out-of-bag score a fair stand-in for cross-validation?** For the forest, `oob_score_` = 0.9445 and the CV mean = 0.9451. The absolute difference is 0.0006, which is not larger than the std of the forest's five fold scores (0.0114), so the two numbers agree. The test accuracy is p = 0.9515 with standard error √(p(1 − p)/n_test) = 0.0066. The OOB score differs from it by 0.0070 (about 1.1 standard errors) and the CV mean by 0.0063 (about 1.0 standard error), so both estimates are close to the test accuracy. The out-of-bag score is a fair, free substitute for cross-validation here.